In [1]:
!gdown 141MgG4CC7XffVH32hQy7lQ0PKCafskji
!mkdir -p hmdb51_data
!unzip -q HMDB51.zip -d hmdb51_data

Downloading...
From (original): https://drive.google.com/uc?id=141MgG4CC7XffVH32hQy7lQ0PKCafskji
From (redirected): https://drive.google.com/uc?id=141MgG4CC7XffVH32hQy7lQ0PKCafskji&confirm=t&uuid=88c0841a-6cbe-4615-bce5-3a26ae7872ab
To: /content/HMDB51.zip
100% 3.37G/3.37G [00:50<00:00, 66.2MB/s]
replace hmdb51_data/brush_hair/April_09_brush_hair_u_nm_np1_ba_goo_0/10000.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


# 1. EDA

# ViT

In [2]:
from __future__ import annotations
import math
import random
from pathlib import Path
from typing import Optional, Tuple, Dict, List
import re

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.transforms.functional as TF

from tqdm.auto import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import timm

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
class VideoTransform:
    def __init__(self, mode="train", img_size=224):
        self.is_train = (mode=='train')
        self.image_size = img_size
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]

    def __call__(self, frames: torch.Tensor) -> torch.Tensor:
        # frames: [T, C, H, W]
        if self.is_train:
            # random r3siz3d crop
            h, w = frames.shape[-2:]
            scale = random.uniform(0.8, 1.0)
            new_h, new_w = int(h*scale), int(w*scale)
            frames = TF.resize(frames, [new_h, new_w], interpolation=transforms.InterpolationMode.BILINEAR)

            # random crop
            i = random.randint(0, max(0, new_h - self.image_size))
            j = random.randint(0, max(0, new_w - self.image_size))
            frames = TF.crop(frames, i, j, min(self.image_size, new_h), min(self.image_size, new_w))
            frames = TF.resize(frames, [self.image_size, self.image_size], interpolation=transforms.InterpolationMode.BILINEAR)

            # horizontal flip (left <--> right)
            if random.random() < 0.5:
                frames = TF.hflip(frames)

        else: #val
            frames = TF.resize(frames, [self.image_size, self.image_size], interpolation=transforms.InterpolationMode.BILINEAR)

        normalized = [TF.normalize(frame, self.mean, self.std) for frame in frames]
        return torch.stack(normalized)

In [4]:
%ls hmdb51_data/brush_hair/April_09_brush_hair_u_nm_np1_ba_goo_1/

10000.jpg  10012.jpg  10024.jpg  10036.jpg  10048.jpg  10060.jpg  10072.jpg
10001.jpg  10013.jpg  10025.jpg  10037.jpg  10049.jpg  10061.jpg  10073.jpg
10002.jpg  10014.jpg  10026.jpg  10038.jpg  10050.jpg  10062.jpg  10074.jpg
10003.jpg  10015.jpg  10027.jpg  10039.jpg  10051.jpg  10063.jpg  10075.jpg
10004.jpg  10016.jpg  10028.jpg  10040.jpg  10052.jpg  10064.jpg  10076.jpg
10005.jpg  10017.jpg  10029.jpg  10041.jpg  10053.jpg  10065.jpg  10077.jpg
10006.jpg  10018.jpg  10030.jpg  10042.jpg  10054.jpg  10066.jpg  10078.jpg
10007.jpg  10019.jpg  10031.jpg  10043.jpg  10055.jpg  10067.jpg
10008.jpg  10020.jpg  10032.jpg  10044.jpg  10056.jpg  10068.jpg
10009.jpg  10021.jpg  10033.jpg  10045.jpg  10057.jpg  10069.jpg
10010.jpg  10022.jpg  10034.jpg  10046.jpg  10058.jpg  10070.jpg
10011.jpg  10023.jpg  10035.jpg  10047.jpg  10059.jpg  10071.jpg


In [5]:
class HMDB51Dataset(Dataset):
    def __init__(self, root: str, split: str, num_frames: int = 16, frame_stride: int = 1,
                 image_size: int = 224, val_ratio: float = 0.1, seed: int = 42):
        super().__init__()

        self.root = Path(root)
        if not self.root.is_dir():
            raise FileNotFoundError(f"Data root not found: {self.root}")

        self.classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        if not self.classes:
            raise RuntimeError(f"No class folders in {self.root}")

        self.class_to_id = {name: id for id, name in enumerate(self.classes)}

        # group videos by (class, base_video_name)
        grouped_samples: Dict[Tuple[str, str], List[Tuple[List[Path], int]]] = {}
        for cls in self.classes:
            cls_dir = self.root / cls
            for video_dir in sorted([d for d in cls_dir.iterdir() if d.is_dir()]):
                frame_paths = sorted([p for p in video_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
                if not frame_paths:
                    continue
                group_key = (cls, self._base_video_name(video_dir.name))
                grouped_samples.setdefault(group_key, []).append((frame_paths, self.class_to_id[cls]))

        if not grouped_samples:
            raise RuntimeError(f"No frame folders found inside {self.root}")

        # split groups
        group_values = list(grouped_samples.values())
        rng = np.random.RandomState(seed)
        group_indices = np.arange(len(group_values))
        rng.shuffle(group_indices)
        split_point = int(len(group_indices) * (1-val_ratio))

        if split == "train":
            selected_groups = group_indices[:split_point]
        elif split in ["val", "test"]:
            selected_groups = group_indices[split_point:]
        else:
            raise ValueError(f"Unknown split: {split}")

        samples: List[Tuple[List[Path], int]] = []

        for id in selected_groups:
            samples.extend(group_values[int(id)])

        if not samples:
            raise RuntimeError(f"Selected split has no samples, adjust ratio or check data folders")

        self.samples = samples
        self.split = split
        self.num_frames = num_frames
        self.frame_stride = max(1, frame_stride)
        self.transform = VideoTransform(mode="train" if split == "train" else "val", img_size=image_size)
        self.to_tensor = transforms.ToTensor()

    def __len__(self) -> int:
        return len(self.samples)

    def _select_indices(self, total: int) -> torch.Tensor:
        if total <= 0:
            raise ValueError("Video folder has no frames")

        if total == 1:
            return torch.zeros(self.num_frames, dtype=torch.long)

        steps = max(self.num_frames*self.frame_stride, self.num_frames)
        grid = torch.linspace(0, total-1, steps=steps)
        ids = grid[:: self.frame_stride].long()

        if ids.numel() < self.num_frames:
            pad = ids.new_full((self.num_frames - ids.numel(),), ids[-1].item())
            ids = torch.cat([ids, pad], dim=0)

        return ids[:self.num_frames]

    @staticmethod
    def _base_video_name(name: str) -> str:
        match = re.match(r"(.+)_\d+$", name)
        return match.group(1) if match else name

    def __getitem__(self, id: int) -> Tuple[torch.Tensor, int]:
        frame_paths, label = self.samples[id]
        total = len(frame_paths)
        ids = self._select_indices(total)

        frames = []
        for i in ids:
            path = frame_paths[int(i.item())]
            with Image.open(path) as img:
                img = img.convert("RGB")
                frames.append(self.to_tensor(img))

        video = torch.stack(frames)
        video = self.transform(video)
        return video, label

def collate_fn(batch: List[Tuple[torch.Tensor, int]]) -> Tuple[torch.Tensor, torch.Tensor]:
    videos = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    return videos, labels

## LOAD DATASET

In [6]:
DATA_ROOT = './hmdb51_data'
VAL_RATIO = 0.1
SEED=42
NUM_FRAMES=16
FRAME_STRIDE=2
IMG_SIZE=224
BATCH_SIZE=4
NUM_WORKERS=4

In [7]:
train_dataset = HMDB51Dataset(
    root=DATA_ROOT,
    split="train",
    num_frames=NUM_FRAMES,
    frame_stride=FRAME_STRIDE,
    image_size=IMG_SIZE,
    val_ratio=VAL_RATIO,
    seed=SEED,
)

val_dataset = HMDB51Dataset(
    root=DATA_ROOT,
    split="val",
    num_frames=NUM_FRAMES,
    frame_stride=FRAME_STRIDE,
    image_size=IMG_SIZE,
    val_ratio=VAL_RATIO,
    seed=SEED,
)

In [8]:
print(f"Train clips: {len(train_dataset)}")
print(f"Train Class count: {len(train_dataset.classes)}")

print(f"Val clips: {len(val_dataset)}")
print(f"Val Class count: {len(val_dataset.classes)}")

Train clips: 6121
Train Class count: 51
Val clips: 645
Val Class count: 51


In [9]:
sample_vid, sample_label = train_dataset[0]
print(f"Train sample: {sample_vid.shape} - {train_dataset.classes[sample_label]}")

Train sample: torch.Size([16, 3, 224, 224]) - catch


In [10]:
sample_vid, sample_label = val_dataset[0]
print(f"Val sample: {sample_vid.shape} - {val_dataset.classes[sample_label]}")

Val sample: torch.Size([16, 3, 224, 224]) - smoke


## DATALOADER

In [11]:
train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=NUM_WORKERS,
                          collate_fn=collate_fn)

val_loader = DataLoader(val_dataset,
                        BATCH_SIZE,
                        shuffle=False,
                        num_workers=NUM_WORKERS,
                        collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 766
Val batches: 81


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## ViT Architecture

In [12]:
class PatchEmbedding(nn.Module):
    """Convert image to patch embeddings"""
    def __init__(self, image_size: int, patch_size: int, in_channels: int, embed_dim: int):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size,
                              stride=patch_size)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """(B, C, H, W) --> (B, num_patches, embed_dim)"""
        x = self.proj(x) # (B, embed_dim, num_patches_on_1_axis, num_patches_on_1_axis)
        x = x.flatten(2) # (B, embed_dim, num_patches)
        x = x.transpose(1, 2) # (B, num_patches, embed_dim)
        return x


In [13]:
x_sim = torch.ones(2, 3, 224, 224)
proj_layer = nn.Conv2d(3, 768, 16,16)

x_out = proj_layer(x_sim)
print(x_out.shape)
print(x_out.flatten(2).transpose(1,2).shape)

torch.Size([2, 768, 14, 14])
torch.Size([2, 196, 768])


In [14]:
class Attention(nn.Module):
    """Multi-head Self-Attention"""
    def __init__(self, dim: int, num_heads: int = 12, attn_drop: float = 0.0, proj_drop: float = 0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim//num_heads
        self.scale = self.head_dim ** (-0.5)

        self.qkv = nn.Linear(dim, dim*3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """(B, N, dim) --> (B, N, dim)"""
        B, N, C = x.shape

        #QKV projection
        qkv = self.qkv(x) # (B, N, dim*3)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4) # (3, B, num_heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention Weights
        attn = (q @ k.transpose(-2, -1))
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # Attention Output
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [15]:
class MLP(nn.Module):
    def __init__(self, in_features: int, hidden_features: int = None, out_features: int = None, drop:float = 0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)

        x = self.fc2(x)
        x = self.drop(x)

        return x

In [16]:
class TransformerBlock(nn.Module):
    """Transformer Block: Attention + MLP"""
    def __init__(self, dim: int, num_heads: int = 12, mlp_ratio: float = 4.0,
                 drop: float = 0.0, attn_drop: float = 0.0):
        super().__init__()

        # attention
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads, attn_drop, drop)

        # mlp
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = MLP(in_features=dim,
                       hidden_features=mlp_hidden,
                       out_features=dim,
                       drop=drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # residual attention
        x = x + self.attn(self.norm1(x))

        #residual mlp
        x = x + self.mlp(self.norm2(x))
        return x

In [17]:
class ViTBase(nn.Module):
    """ViT-Base from scratch (vit_base_patch16_224 architecture)

    Config:
    - Image size: 224
    - Patch size: 16
    - Embed dim: 768
    - Depth: 12
    - Num heads: 12
    - MLP ratio: 4.0
    """
    def __init__(self, image_size: int = 224, patch_size: int = 16, in_channels: int = 3,
                 embed_dim: int = 768, depth: int = 12, num_heads: int = 12,
                 mlp_ratio: float = 4.0, drop_rate: float = 0.0, attn_drop_rate: float = 0.0):
        super().__init__()

        self.embed_dim = embed_dim

        # patch embedding
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        # class token
        self.cls_token = nn.Parameter(torch.zeros(1,1, embed_dim))

        # positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches+1, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)

        # transformer Blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop_rate)
            for _ in range(depth)
        ])

        # layer norm
        self.norm = nn.LayerNorm(embed_dim)

        # initialize weights
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """Extract Feature w/o Classification Head"""
        B = x.shape[0]

        # patch embedding
        x = self.patch_embed(x) # (B, num_patches, embed_dim)

        # add class token
        cls_tokens = self.cls_token.expand(B, -1, -1) # number of cls tokens = batch_size
        x = torch.cat([cls_tokens, x], dim = 1) # (B, num_patches+1, embed_dim)

        # add positional embedding
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # feed through Transformer Blocks
        for block in self.blocks:
            x = block(x)

        # layer norm
        x = self.norm(x)

        return x # still (B, num_patches+1, embed_dim)


In [18]:
class BasicViT(nn.Module):
    """BasicViT for video classification

    Pipeline:
    1. ViT-Base backbone (from scratch)
    2. Process each frame -> extract features
    3. Mean pooling across time (BOTTLENECK!)
    4. Classification head
    """
    def __init__(self, num_classes: int = 51):
        super().__init__()

        # ViT Backbone as Feature Extractor
        self.vit = ViTBase(
            image_size=224,
            patch_size=16,
            in_channels=3,
            embed_dim=768,
            depth=12,
            num_heads=12,
            mlp_ratio=4.0,
            drop_rate=0.0,
            attn_drop_rate=0.0
        )

        self.embed_dim = self.vit.embed_dim

        # Classification head
        self.head = nn.Linear(self.embed_dim, num_classes)

    def forward(self, video: torch.Tensor) -> torch.Tensor:
        """
        Input: (B, T, C, H, W) where T is number of frames/video (usually 16)
        Output: (B, num_classes)
        """
        B, T, C, H, W = video.shape

        # feed each frame through ViT
        x = video.reshape(B*T, C, H, W)
        x = self.vit.forward_features(x) # (B*T, num_patches+1, 768)
        x = x[:, 0, :] # extract cls token (B*T, 768)

        # Mean Pooling across frames (Caution: LOSE temporal info!)
        x = x.reshape(B, T, self.embed_dim) # (B, T, 768)
        x = x.mean(dim=1)

        # classification
        logits = self.head(x) # (B, num_classes)
        return logits

In [19]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BasicViT(num_classes=len(train_dataset.classes)).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())

print(f"   Model created")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"   Architecture: ViT-Base from scratch")
print(f"   Status: Untrained (from scratch)")

   Model created
   Total parameters: 85,837,875 (85.84M)
   Architecture: ViT-Base from scratch
   Status: Untrained (from scratch)


In [20]:
def load_pretrained_vit_checkpoint(model, device):
    """Load pretrained ViT-Base weights from timm into our custom ViT model"""
    # load timm ViT-base pretrained
    timm_model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
    timm_model = timm_model.to(device)
    timm_state = timm_model.state_dict()

    # Map & Load weights
    custom_state = model.vit.state_dict()
    pretrained_keys = 0

    for key in custom_state.keys():
        if key in timm_state:
            custom_state[key] = timm_state[key]
            pretrained_keys += 1

    model.vit.load_state_dict(custom_state, strict=False)

    print(f"Loaded pretrained ViT-Base ckpt from timm")
    print(f"    Keys loaded: {pretrained_keys}/{len(custom_state)}")
    print(f"    Status: Ready for fine-tuning")

    return model

In [21]:
model = load_pretrained_vit_checkpoint(model, DEVICE)

Loaded pretrained ViT-Base ckpt from timm
    Keys loaded: 150/150
    Status: Ready for fine-tuning


## Training

Train Function

In [22]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """

    Return:
        epoch_loss: float, epoch_acc: float
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training...")
    for videos, labels in pbar:
        videos = videos.to(device)
        labels = labels.to(device)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.bacward()
        optimizer.step()

        # metrics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        pbar.setpostfix({'loss': f'{total_loss / (pbar.n + 1):.4f}',
                         'acc': f'{correct / total:.4f}'})

    epoch_loss = total_loss / len(dataloader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validating....")
        for videos, labels in pbar:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)
            loss = criterion(outputs, labels)

            # metrics
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

            pbar.setpostfix({'loss': f'{total_loss / (pbar.n + 1):.4f}',
                           'acc': f'{correct / total:.4f}'})

    epoch_loss = total_loss / len(dataloader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


In [23]:
def denormalize(frames):
    """Denormalize for visualizing"""
    frames = frames.clone()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1) # mean & std of ImageNet
    std = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)
    frames = frames * std + mean
    return frames.clamp(0,1)

Train Setup

In [24]:
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE,
                             weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Training setup:")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Trainable parameters: {trainable_params:,} (ALL params)")
print(f"   Optimizer: Adam")
print(f"   Scheduler: CosineAnnealing")

   Training setup:
   Epochs: 20
   Learning rate: 0.0001
   Trainable parameters: 85,837,875 (ALL params)
   Optimizer: Adam
   Scheduler: CosineAnnealing


TRAININGGGG

In [25]:
import time

In [26]:
start = time.time()
for i in range(1000):
    continue
time.time() - start

0.00011920928955078125

In [27]:
history = {'train_loss': [], 'train_acc': [],
           'val_loss': [], 'val_acc': [],
           'train_time': []}
best_acc = 0.0
best_model_path = 'basicvit_best.pt'

print(f"\n{'='*63}")
print(f"Fine-tuning BasicViT for {EPOCHS} epochs")
print(f"\n{'='*63}")


for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    duration = time.time() - start

    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)


    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['train_time'].append(duration)

    print(f"  -Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    print(f"  -Val:   Loss={val_loss:.4f}, Acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"    Best model saved with acc = {best_acc:.4f}")

    scheduler.step()

print(f"\n{'='*63}")
print(f"Training complete!")
print(f"Best val accuracy: {best_acc:.4f}")
print(f"\n{'='*63}")



Fine-tuning BasicViT for 20 epochs

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Training...:   0%|          | 0/766 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 296.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 285.81 MiB is free. Including non-PyTorch memory, this process has 14.28 GiB memory in use. Of the allocated memory 13.90 GiB is allocated by PyTorch, and 258.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Plot results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,6))

# plot loss
axes[0].plot(history['train_loss'], label='Train', marker = 'o')
axes[0].plot(history['val_loss'], label='Val', marker='s')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# plot accuracy
axes[1].plot(history['train_acc'], label='Train', marker = 'o')
axes[1].plot(history['val_acc'], label='Val', marker='s')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0,1])

plt.tight_layout()
plt.savefig("basic_vit_training.png", dpi=100, bbox_inches='tight')
print("Plot saved!")
plt.show()

# LS-ViT


# other approaches